# 1) Importacion de CVS (mokadata) para generar las tablas

In [ ]:
import pandas as pd

# Reemplaza estas URLs con los enlaces de descarga directa de tus archivos CSV
# Los links cargados corresponder son mokadata
item_url = 'https://drive.google.com/uc?export=download&id=19MXqb7MRy9cp9LrpSDyD-npPNTb-bZ3v'
presupuestos_url = 'https://drive.google.com/uc?export=download&id=1NpWrZBxYOqT6FPbGJqIMD21q468y6A47'
detalles_url = 'https://drive.google.com/uc?export=download&id=1WLzIwb5JBr_msd8f8seDKh5PkQFgKEUJ'

# Pandas lee los CSVs directamente desde la URL pública
df_items = pd.read_csv(item_url)
df_presupuestos = pd.read_csv(presupuestos_url)
df_detalles = pd.read_csv(detalles_url)

print("¡DataFrames cargados exitosamente desde la web!")

¡DataFrames cargados exitosamente desde la web!


# 2) Creamos una sola tabla unificando Presupuestos, Detalles e Items para un vistazo general.

In [ ]:

# Habiendo cargado los CSVs en DataFrames: df_presupuestos, df_presupuestos, df_presupuestos

# 1. Cruzamos PRESUPUESTO con su DETALLE (Relación Uno a Muchos)
# Usamos left_on y right_on porque los nombres de las columnas ID varían entre tablas
print("Cruzo presupuestos con sus detalles...")
prep_con_detalles = pd.merge(
    df_presupuestos,
    df_detalles,
    left_on='id',
    right_on='presupuesto_id',
    suffixes=('_presupuesto', '_detalle')
)

# 2. A ese resultado, le cruzamos la tabla de ITEM para saber el NOMBRE y DESCRIPCIÓN del producto vendido
print("Agrego la información de los ítems catalogados...")
df_consolidado_final = pd.merge(
    prep_con_detalles,
    df_items,
    left_on='item_id',
    right_on='id',
    suffixes=('', '_master_item')
)

# ¡Listo! Tenés una sola tabla con toda la historia económica de la app
print("Columnas resultantes listas para analizar:")
print(df_consolidado_final.columns.tolist(),"\n\n")
display(df_consolidado_final.head())



Cruzo presupuestos con sus detalles...
Agrego la información de los ítems catalogados...
Columnas resultantes listas para analizar:
['id_presupuesto', 'usuario_id', 'cliente_id', 'fecha_creacion', 'subtotal_presupuesto', 'total', 'estado', 'pdf_url', 'fecha_vencimiento', 'created_at_presupuesto', 'updated_at_presupuesto', 'deleted_at', 'id_detalle', 'presupuesto_id', 'item_id', 'cantidad', 'precio_unitario', 'subtotal_detalle', 'created_at_detalle', 'updated_at_detalle', 'id', 'usuario_id_master_item', 'nombre', 'descripcion', 'precio', 'activo', 'created_at', 'updated_at', 'deleted_at_master_item'] 




,id_presupuesto,usuario_id,cliente_id,fecha_creacion,subtotal_presupuesto,total,estado,pdf_url,fecha_vencimiento,created_at_presupuesto,...,updated_at_detalle,id,usuario_id_master_item,nombre,descripcion,precio,activo,created_at,updated_at,deleted_at_master_item
0,PRP-001,USR-001,CLI-001,2026-06-01,21000,21000,Guardado,https://storage.app/pdf/p1.pdf,2026-06-15,2026-06-01 10:00:00,...,2026-06-01 10:00:00,ITM-001,USR-001,Caño Termofusión Viga,Caño de agua de 20mm marca IPS,2500,True,2026-05-01 9:30:00,2026-05-01 9:30:00,NaN
1,PRP-001,USR-001,CLI-001,2026-06-01,21000,21000,Guardado,https://storage.app/pdf/p1.pdf,2026-06-15,2026-06-01 10:00:00,...,2026-06-01 10:00:00,ITM-002,USR-001,Mano de Obra Plomería,Servicio de reparación por hora,8000,True,2026-05-01 9:35:00,2026-05-01 9:35:00,NaN
2,PRP-002,USR-001,CLI-002,2026-06-02,2500,2500,Borrador,NaN,2026-06-17,2026-06-02 15:20:00,...,2026-06-02 15:20:00,ITM-001,USR-001,Caño Termofusión Viga,Caño de agua de 20mm marca IPS,2500,True,2026-05-01 9:30:00,2026-05-01 9:30:00,NaN
3,PRP-003,USR-001,CLI-003,2026-06-03,32200,32200,Guardado,https://storage.app/pdf/p3.pdf,2026-06-18,2026-06-03 11:45:00,...,2026-06-03 11:45:00,ITM-002,USR-001,Mano de Obra Plomería,Servicio de reparación por hora,8000,True,2026-05-01 9:35:00,2026-05-01 9:35:00,NaN
4,PRP-003,USR-001,CLI-003,2026-06-03,32200,32200,Guardado,https://storage.app/pdf/p3.pdf,2026-06-18,2026-06-03 11:45:00,...,2026-06-03 11:45:00,ITM-003,USR-001,Llave de Paso 3/4,Llave de paso esférica de bronce,6500,True,2026-05-01 9:40:00,2026-05-01 9:40:00,NaN


# 3) Empezamos con el tracking básico para ver métricas

In [ ]:
# Tracking básico: Métricas acumuladas del sistema
total_presupuestos = df_presupuestos['id'].nunique()
monto_total_generado = df_presupuestos['total'].sum()

# Tracking específico por Estado (para ver qué Estado predomina dentro de la app)
tracking_por_estado = df_presupuestos.groupby('estado').agg(
    Cantidad=('id', 'count'),
    Monto_Total=('total', 'sum')
).reset_index()

print(f"--- PANEL DE CONTROL DE TRACKING BÁSICO ---")
print(f"Total de Presupuestos Creados en la App: {total_presupuestos}")
print(f"Total de Montos Financieros Generados: ${monto_total_generado:,.2f}")
print("\nDesglose por Estado de Presupuesto:")
print(tracking_por_estado)

--- PANEL DE CONTROL DE TRACKING BÁSICO ---
Total de Presupuestos Creados en la App: 20
Total de Montos Financieros Generados: $1,136,200.00

Desglose por Estado de Presupuesto:
     estado  Cantidad  Monto_Total
0  Borrador         5        79900
1  Guardado        15      1056300


In [ ]:
# Tracking diario de creación e ingresos (aquí vemos la cantidad de Presupuestos creados x día)
tracking_diario = df_presupuestos.groupby('fecha_creacion').agg(
    Presupuestos_Creados=('id', 'count'),
    Montos_Del_Dia=('total', 'sum')
).reset_index().sort_values('fecha_creacion')

print("\nEvolución Diaria del Tracking (Línea de tiempo):")
print(tracking_diario)


Evolución Diaria del Tracking (Línea de tiempo):
  fecha_creacion  Presupuestos_Creados  Montos_Del_Dia
0     2026-06-01                     1           21000
1     2026-06-02                     2           57500
2     2026-06-03                     2           36400
3     2026-06-04                     2           53500
4     2026-06-05                     3          223400
5     2026-06-06                     3          253500
6     2026-06-07                     3           55600
7     2026-06-08                     2          159300
8     2026-06-09                     1           41000
9     2026-06-10                     1          235000


# 4) Simulación extracción eventos de Firebase (presup generados/visualizados)


In [ ]:
# Simulación del formato real con el que Firebase exporta sus eventos
firebase_events_mock = [
    {
        "event_name": "presupuesto_creado",
        "event_timestamp": "2026-07-22 14:30:00",
        "user_id": "USR-004",
        "event_params": {"monto": 470000, "estado": "Guardado"}
    },
    {
        "event_name": "generacion_pdf",
        "event_timestamp": "2026-07-22 14:32:15",
        "user_id": "USR-004",
        "event_params": {"presupuesto_id": "PRP-010", "formato": "pdf"}
    },
    {
        "event_name": "visualizacion_historial",
        "event_timestamp": "2026-07-22 15:00:10",
        "user_id": "USR-002",
        "event_params": {"origen": "home"}
    }
]

df_events = pd.DataFrame(firebase_events_mock)
display(df_events)


,event_name,event_timestamp,user_id,event_params
0,presupuesto_creado,2026-07-22 14:30:00,USR-004,"{'monto': 470000, 'estado': 'Guardado'}"
1,generacion_pdf,2026-07-22 14:32:15,USR-004,"{'presupuesto_id': 'PRP-010', 'formato': 'pdf'}"
2,visualizacion_historial,2026-07-22 15:00:10,USR-002,{'origen': 'home'}


# 5) Prueba metricas

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Fijamos semilla para reusabilidad
random.seed(42)

# 1. BASE DE DATOS SINTÉTICA DE PRESUPUESTOS (Para vincular IDs reales)
# Si ya tenés 'df_presupuestos' en memoria, el script usará tus datos reales
try:
    presupuestos_ref = df_presupuestos[['id', 'usuario_id', 'total', 'created_at']].to_dict('records')
    print("✓ Conectado exitosamente con la tabla de PRESUPUESTOS cargada en memoria.")
except NameError:
    # Sino está cargado el DS, el script usa estos datos de respaldo:
    presupuestos_ref = [
        {'id': 'PRP-001', 'usuario_id': 'USR-001', 'total': 12000.0, 'created_at': '2026-07-01 10:00:00'},
        {'id': 'PRP-002', 'usuario_id': 'USR-002', 'total': 45000.0, 'created_at': '2026-07-05 11:30:00'},
        {'id': 'PRP-003', 'usuario_id': 'USR-004', 'total': 470000.0, 'created_at': '2026-07-10 15:20:00'},
        {'id': 'PRP-004', 'usuario_id': 'USR-004', 'total': 79900.0, 'created_at': '2026-07-15 09:15:00'},
        {'id': 'PRP-005', 'usuario_id': 'USR-003', 'total': 32000.0, 'created_at': '2026-07-18 18:45:00'},
    ]
    print("! Generando matriz sintética de referencia de Presupuestos.")

# Lista global donde acumularemos todos los eventos de comportamiento
log_eventos = []

# 2. GENERACIÓN DE EVENTOS VINCULADOS A PRESUPUESTOS (Creación y PDF)
for p in presupuestos_ref:
    fecha_creacion = datetime.strptime(str(p['created_at'])[:19], '%Y-%m-%d %H:%M:%S')

    # --- EVENTO 1: Creación de Presupuesto ---
    log_eventos.append({
        'event_id': f"EVT-{len(log_eventos)+1:05d}",
        'event_name': 'presupuesto_creado',
        'timestamp': fecha_creacion.strftime('%Y-%m-%d %H:%M:%S'),
        'user_id': p['usuario_id'],
        'presupuesto_id': p['id'],
        'monto': p['total'],
        'pantalla': 'formulario_presupuesto',
        'dispositivo': random.choice(['Android', 'iOS', 'Web'])
    })

    # --- EVENTO 2: Generación de PDF (Simulando que el 80% de los presupuestos se descargan) ---
    if random.random() < 0.8:
        # La descarga ocurriría entre 1 y 12 minutos después de crearlo
        fecha_pdf = fecha_creacion + timedelta(minutes=random.randint(1, 12))
        log_eventos.append({
            'event_id': f"EVT-{len(log_eventos)+1:05d}",
            'event_name': 'generacion_pdf',
            'timestamp': fecha_pdf.strftime('%Y-%m-%d %H:%M:%S'),
            'user_id': p['usuario_id'],
            'presupuesto_id': p['id'],
            'monto': p['total'],
            'pantalla': 'detalle_presupuesto',
            'dispositivo': random.choice(['Android', 'iOS', 'Web'])
        })

# 3. GENERACIÓN DE EVENTOS DE NAVEGACIÓN (Visualización de Historial)
# Los usuarios revisan el historial múltiples veces al mes sin crear necesariamente un presupuesto
usuarios_unicos = list(set(p['usuario_id'] for p in presupuestos_ref))

for user in usuarios_unicos:
    # Simulamos entre 3 y 6 visitas al historial por usuario durante el mes
    num_visitas = random.randint(3, 6)
    for _ in range(num_visitas):
        dia = random.randint(1, 20)
        hora = random.randint(8, 21)
        minuto = random.randint(0, 59)
        fecha_historial = datetime(2026, 7, dia, hora, minuto)

        log_eventos.append({
            'event_id': f"EVT-{len(log_eventos)+1:05d}",
            'event_name': 'visualizacion_historial',
            'timestamp': fecha_historial.strftime('%Y-%m-%d %H:%M:%S'),
            'user_id': user,
            'presupuesto_id': None,  # No está asociado a un presupuesto específico
            'monto': None,
            'pantalla': 'home_dashboard',
            'dispositivo': random.choice(['Android', 'iOS', 'Web'])
        })

# 4. CONSOLIDACIÓN EN DATAFRAME Y ORDENAMIENTO CRONOLÓGICO
df_log_eventos = pd.DataFrame(log_eventos)
df_log_eventos['timestamp'] = pd.to_datetime(df_log_eventos['timestamp'])
df_log_eventos = df_log_eventos.sort_values('timestamp').reset_index(drop=True)

# 5. RESUMEN EJECUTIVO DEL TRACKING
print("\n" + "="*50)
print("  TABLA DE TRACKING DE EVENTOS DE COMPORTAMIENTO")
print("="*50)
print(f"Total de eventos capturados: {len(df_log_eventos)}")
print("\nDesglose de conteo por Evento:")
print(df_log_eventos['event_name'].value_counts())
print("\n\nMuestra de los primeros 5 registros:")
df_log_eventos.head()

✓ Conectado exitosamente con la tabla de PRESUPUESTOS cargada en memoria.

  TABLA DE TRACKING DE EVENTOS DE COMPORTAMIENTO
Total de eventos capturados: 63

Desglose de conteo por Evento:
event_name
visualizacion_historial    26
presupuesto_creado         20
generacion_pdf             17
Name: count, dtype: int64


Muestra de los primeros 5 registros:


,event_id,event_name,timestamp,user_id,presupuesto_id,monto,pantalla,dispositivo
0,EVT-00001,presupuesto_creado,2026-06-01 10:00:00,USR-001,PRP-001,21000.0,formulario_presupuesto,Web
1,EVT-00002,generacion_pdf,2026-06-01 10:12:00,USR-001,PRP-001,21000.0,detalle_presupuesto,iOS
2,EVT-00009,presupuesto_creado,2026-06-02 10:30:00,USR-002,PRP-005,55000.0,formulario_presupuesto,Android
3,EVT-00010,generacion_pdf,2026-06-02 10:40:00,USR-002,PRP-005,55000.0,detalle_presupuesto,Android
4,EVT-00003,presupuesto_creado,2026-06-02 15:20:00,USR-001,PRP-002,2500.0,formulario_presupuesto,Android


# 6) Paso de métricas a Dash (Looker)


In [ ]:
from google.colab import files

# 1. Agregamos las métricas de comportamiento a nivel de Presupuesto y Usuario
eventos_por_presupuesto = df_log_eventos[df_log_eventos['presupuesto_id'].notnull()].groupby('presupuesto_id').agg(
    descargo_pdf=('event_name', lambda x: 1 if 'generacion_pdf' in list(x) else 0)
).reset_index()

eventos_por_usuario = df_log_eventos.groupby('user_id').agg(
    total_visitas_historial=('event_name', lambda x: (x == 'visualizacion_historial').sum())
).reset_index()

# 2. Unimos todo con la tabla principal de Presupuestos (Left Join)
df_dashboard = df_presupuestos.copy()

# Vinculamos si el presupuesto tuvo PDF
df_dashboard = df_dashboard.merge(eventos_por_presupuesto, left_on='id', right_on='presupuesto_id', how='left')
df_dashboard['descargo_pdf'] = df_dashboard['descargo_pdf'].fillna(0).astype(int)

# Vinculamos la actividad del usuario
df_dashboard = df_dashboard.merge(eventos_por_usuario, left_on='usuario_id', right_on='user_id', how='left')
df_dashboard['total_visitas_historial'] = df_dashboard['total_visitas_historial'].fillna(0).astype(int)

# 3. Limpieza de columnas sobrantes
df_dashboard = df_dashboard.drop(columns=['presupuesto_id', 'user_id'], errors='ignore')

# 4. EXPORTACIÓN A CSV PARA LOOKER STUDIO
df_dashboard.to_csv('dashboard_innova_mvp.csv', index=False)
files.download('dashboard_innova_mvp.csv')

print("✓ ¡Dataset 'dashboard_innova_mvp.csv' generado con éxito!")
print(f"Dimensiones de la tabla final: {df_dashboard.shape[0]} filas x {df_dashboard.shape[1]} columnas")
df_dashboard.head()
print("Ver total de la columna 'Total'")
print(df_dashboard['total'].sum())



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ ¡Dataset 'dashboard_innova_mvp.csv' generado con éxito!
Dimensiones de la tabla final: 20 filas x 14 columnas
Ver total de la columna 'Total'
1136200
